GOAL: Need to design a cheap model specific to the feild that is able to do better analysis than a powerful general purpose model. 

Nowdays, the pace at which you are able to implement and improve new ML models is almost instant and much more proliferated than the process of the past. It is more important to undrstand today how to use them rather than what goes into them underneath the hood for real world applications AKA **Business** applicaitons.

While you are figuring out how to get comfortable once again with Tableu, look into what we actually want to get on the graph. After you understand that work close with your teamates to make sure they are able to help out with the graph. FOCUS your energy on the most important topics that are needed for the graph first.

1. The Cultural (categorical, review style of a select areal) z-score (relative score to area)
2. Certainty rating of if a review is actually likely to be the average rating that it has.
3. Sentiment analysis ***(A 4 star review with negative sentiment means user isn't fully satisfied)***
(Idea basically trust the review unless the sentiment is conflicting, if conflicting overide with sentiment becasue I think reviews are the best approximation to a degree)

4️⃣ Example workflow

Collect user rating and review text.

Apply sentiment analysis (e.g., using Python’s TextBlob, Vader, or Hugging Face models).

Compute a sentiment polarity score (range -1 to 1).

Compare polarity score vs rating:

High rating but negative sentiment → flag as potential mismatch.

Low rating but positive sentiment → investigate further.

Optionally, adjust average rating using sentiment as a weight.

5️⃣ Benefits (mainly for rating acuracy and fake/accidental/even_bot_ratings)

Detects inconsistencies between numeric ratings and actual user feelings.

Improves recommendation systems by using more accurate satisfaction indicators.

Helps identify fake reviews or accidental ratings.

💡 Insight: Ratings are quantitative, sentiment is qualitative. Using both together gives a richer picture of user satisfaction.

For #1. you can start with just categorical z-scores and see IF THERE IS AN ACTUAL DIFFERENCE IN HOW IT APPEARS JUST LIKE AURORA'S BAR GRAPHS. Becasue right now I don't know wether to trust if Aiden's code works for future enhancements.

**Worst case you could do a visual overview of the state's frequency of ratings and draw your own interpretation of "Cultures". Then make individual normalizations for however many "Cultural Regions" That there are.**

(but lets first start with only categorical z-scores since that will be most impactful and easiest to implement, "Apples to Apples")

In [ ]:
#TODO CONCATINATE CATEGORY TOGETHER FOR OVERLAPING CATEGORIES then derrive z-scores for a company within each category. 
#TODO CLEAN/ADD confidence intervals for each company for Tableu visualization, permanently closed, <------Base Column to start with
#TODO Start Tableu visualizations and learn what we can and cannot do with the curretn data (Where do we need to feature engineer?)
#TODO Flag conflicting sentiment v.s reviews rows and DECIDE WHAT TO DO WITH THEM. –Treat them differently in models (e.g., reduce their weight).
#TODO After all of the above stuff is done, then figure out how to actually weight the sentiment beacuse TEXT IS SCARCE.
#TODO After After all of the above then think about user profiles but more than likley we wont have time to do this.
#TODO Straight up train a model index <----
#TODO Optimization techniques for tableau to make it smaller and easier to load.

In [ ]:
#TODO Get up a Basic presentable dashboard in Tableu by this weekend, then begin the work to put in the actual data that you want and get the model up and running.

**It looks like there are way too many dental offices do I need to sample my data differently?** ALSO I NEED TO FILTER FOR OUTLIERS IN THE NUMBER OF REVIEWS

Something is wrong with the number of reviews data maybe it is null values? BC VScode and Tableau arent matching up

NO NULL VALUES

In [3]:
import pandas as pd
import numpy as np
import gzip
import json

pd.set_option('display.max_columns', None)

def parse(path):
  g = open('Data/' + path, 'r')
  for l in g:
    yield json.loads(l)

def parse_first_n(path, n=10000):
    g = open('Data/' + path, 'r')
    for i, l in enumerate(g):
        if i >= n:
            break
        yield json.loads(l)

In [ ]:
texas_metadata = pd.DataFrame(parse("meta-Texas.json"))
texas_reviews = pd.DataFrame(parse_first_n("review-Texas.json", n=1000000)) # is not cleaned yet


In [35]:
texas_reviews = texas_reviews.dropna(subset=['rating']) # Drop rows where 'rating' is NaN

In [ ]:
texas_metadata = texas_metadata.dropna(subset=['gmap_id', 'name']).drop_duplicates(subset=['gmap_id'], keep='first')
texas_metadata.head()


,name,address,gmap_id,description,latitude,longitude,category,avg_rating,num_of_reviews,price,hours,MISC,state,relative_results,url
0,Timewise Food Store,"Timewise Food Store, 1600 W Church St, Livings...",0x8638869e6b4e3529:0xe8d257447fe41672,None,30.713368,-94.954344,[Convenience store],4.8,4,None,"[[Thursday, Open 24 hours], [Friday, Open 24 h...","{'Service options': ['In-store shopping', 'Del...",Open 24 hours,"[0x863886bab3f9bb05:0x28a8062d0597dd34, 0x8638...",https://www.google.com/maps/place//data=!4m2!3...
1,Goldstar Transit,"Goldstar Transit, 4415 W Dickson Ln, Little El...",0x864c3748dcc1c12d:0xbf904a61f262cf9b,None,33.159363,-96.975571,[Transportation service],4.5,4,None,"[[Thursday, 6AM–6PM], [Friday, 6AM–6PM], [Satu...",{'Accessibility': ['Wheelchair accessible entr...,Open ⋅ Closes 6PM,"[0x864c374855555555:0x3abb669a098bb235, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...
2,Walmart Pharmacy,"Walmart Pharmacy, 12220 FM 423, Frisco, TX 75033",0x864c3998b8d8dc83:0x57ffabe1e2322320,None,33.179867,-96.883691,"[Pharmacy, Drug store, Medical supply store, V...",3.3,24,$,"[[Thursday, 9AM–9PM], [Friday, 9AM–9PM], [Satu...","{'Service options': ['Curbside pickup', 'In-st...",Open ⋅ Closes 9PM,"[0x864c3999b29e291f:0x2d364c05e88eec13, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...
3,Luminous Logistics,"Luminous Logistics, 3838 W Miller Rd, Garland,...",0x864ea0993bffffff:0xb50b5bb2fccf9d9b,None,32.893678,-96.688611,[Delivery service],2.3,8,None,None,None,None,"[0x864ea09938bb619f:0x1b6902de2a2f3f96, 0x864e...",https://www.google.com/maps/place//data=!4m2!3...
4,Pacesetter Personnel Services,"Pacesetter Personnel Services, 2300 Valley Vie...",0x864e819d99a1ff99:0xeee31cc82854286c,None,32.839795,-97.020987,[Employment agency],2.1,7,None,None,None,None,"[0x864e9d6ea0c9089f:0x6f90f8b0b092af49, 0x864e...",https://www.google.com/maps/place//data=!4m2!3...


In [24]:
texas_metadata = texas_metadata[texas_metadata['state'] != 'Permanently closed']


In [26]:
texas_metadata = texas_metadata[(texas_metadata['latitude'] >= 25.5) & (texas_metadata['latitude'] <= 36.40)
                            & (texas_metadata['longitude'] >= -106.50) & (texas_metadata['longitude'] <= -93.31)]

In [ ]:
#TODO create a confidence interval column for the certaintly of the avg rating
- get the ratings column per each company post merge, and 
figure it out for each company

(416224, 15)

In [36]:
def central_limit_95_percent_confidence_interval(data):
    mean = np.mean(data)
    std_error = np.std(data, ddof=1) / np.sqrt(len(data))
    margin_of_error = 1.96 * std_error
    return mean - margin_of_error, mean + margin_of_error

def individual_company_rating_confidence_interval(gmap_id):
    company_reviews = texas_reviews[texas_reviews['gmap_id'] == gmap_id]['rating']
    return central_limit_95_percent_confidence_interval(company_reviews)

How Can I add a new column that matches up with each business in either the metadata or the merged which displays the confidence interval?

In [ ]:

#Extract all unique category labels from california_metadata

category_df = texas_metadata.copy()
category_df = category_df.explode('category')
#Convert everything to lower case and remove blank space 
category_df['category'] = category_df['category'].str.lower().str.strip()

# # # #Remove tiny categories 
category_counts= category_df.get('category').value_counts()


# Setting threshold for the category 
category_counts.describe()
category_counts_threshold = 35 #median


# Map category_count to each category in the df
category_df['category_count'] = category_df['category'].map(category_counts)
category_df = category_df[category_df.get('category_count') > category_counts_threshold]
category_df # a part of metadata just longer becasue of explode

# I need a mean and std for each 

,name,address,gmap_id,description,latitude,longitude,category,avg_rating,num_of_reviews,price,hours,MISC,state,relative_results,url,category_count
0,Timewise Food Store,"Timewise Food Store, 1600 W Church St, Livings...",0x8638869e6b4e3529:0xe8d257447fe41672,None,30.713368,-94.954344,convenience store,4.8,4,None,"[[Thursday, Open 24 hours], [Friday, Open 24 h...","{'Service options': ['In-store shopping', 'Del...",Open 24 hours,"[0x863886bab3f9bb05:0x28a8062d0597dd34, 0x8638...",https://www.google.com/maps/place//data=!4m2!3...,9092.0
1,Goldstar Transit,"Goldstar Transit, 4415 W Dickson Ln, Little El...",0x864c3748dcc1c12d:0xbf904a61f262cf9b,None,33.159363,-96.975571,transportation service,4.5,4,None,"[[Thursday, 6AM–6PM], [Friday, 6AM–6PM], [Satu...",{'Accessibility': ['Wheelchair accessible entr...,Open ⋅ Closes 6PM,"[0x864c374855555555:0x3abb669a098bb235, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,1160.0
2,Walmart Pharmacy,"Walmart Pharmacy, 12220 FM 423, Frisco, TX 75033",0x864c3998b8d8dc83:0x57ffabe1e2322320,None,33.179867,-96.883691,pharmacy,3.3,24,$,"[[Thursday, 9AM–9PM], [Friday, 9AM–9PM], [Satu...","{'Service options': ['Curbside pickup', 'In-st...",Open ⋅ Closes 9PM,"[0x864c3999b29e291f:0x2d364c05e88eec13, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,1950.0
2,Walmart Pharmacy,"Walmart Pharmacy, 12220 FM 423, Frisco, TX 75033",0x864c3998b8d8dc83:0x57ffabe1e2322320,None,33.179867,-96.883691,drug store,3.3,24,$,"[[Thursday, 9AM–9PM], [Friday, 9AM–9PM], [Satu...","{'Service options': ['Curbside pickup', 'In-st...",Open ⋅ Closes 9PM,"[0x864c3999b29e291f:0x2d364c05e88eec13, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,2933.0
2,Walmart Pharmacy,"Walmart Pharmacy, 12220 FM 423, Frisco, TX 75033",0x864c3998b8d8dc83:0x57ffabe1e2322320,None,33.179867,-96.883691,medical supply store,3.3,24,$,"[[Thursday, 9AM–9PM], [Friday, 9AM–9PM], [Satu...","{'Service options': ['Curbside pickup', 'In-st...",Open ⋅ Closes 9PM,"[0x864c3999b29e291f:0x2d364c05e88eec13, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,456.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
447313,O'Reilly Auto Parts,"O'Reilly Auto Parts, 702 Tennessee Ave, Dalhar...",0x8705b6b8981c266d:0x5770fa9f1b6c0b1a,None,36.058740,-102.514025,car repair and maintenance,4.3,94,None,"[[Wednesday, 7AM–9PM], [Thursday, 7AM–9PM], [F...","{'Service options': ['Curbside pickup', 'In-st...",NaN,"[0x8705b6c7859a9041:0x9107ec50aa4d5609, 0x8705...",https://www.google.com/maps/place//data=!4m2!3...,3356.0
447313,O'Reilly Auto Parts,"O'Reilly Auto Parts, 702 Tennessee Ave, Dalhar...",0x8705b6b8981c266d:0x5770fa9f1b6c0b1a,None,36.058740,-102.514025,car stereo store,4.3,94,None,"[[Wednesday, 7AM–9PM], [Thursday, 7AM–9PM], [F...","{'Service options': ['Curbside pickup', 'In-st...",NaN,"[0x8705b6c7859a9041:0x9107ec50aa4d5609, 0x8705...",https://www.google.com/maps/place//data=!4m2!3...,1140.0
447313,O'Reilly Auto Parts,"O'Reilly Auto Parts, 702 Tennessee Ave, Dalhar...",0x8705b6b8981c266d:0x5770fa9f1b6c0b1a,None,36.058740,-102.514025,motorcycle parts store,4.3,94,None,"[[Wednesday, 7AM–9PM], [Thursday, 7AM–9PM], [F...","{'Service options': ['Curbside pickup', 'In-st...",NaN,"[0x8705b6c7859a9041:0x9107ec50aa4d5609, 0x8705...",https://www.google.com/maps/place//data=!4m2!3...,943.0
447313,O'Reilly Auto Parts,"O'Reilly Auto Parts, 702 Tennessee Ave, Dalhar...",0x8705b6b8981c266d:0x5770fa9f1b6c0b1a,None,36.058740,-102.514025,trailer supply store,4.3,94,None,"[[Wednesday, 7AM–9PM], [Thursday, 7AM–9PM], [F...","{'Service options': ['Curbside pickup', 'In-st...",NaN,"[0x8705b6c7859a9041:0x9107ec50aa4d5609, 0x8705...",https://www.google.com/maps/place//data=!4m2!3...,1465.0


In [29]:

# Calculate mean and standard deviation of each category 
category_stats = category_df.groupby("category")['avg_rating'].agg(['mean', 'std']).reset_index()
category_stats.rename(columns= {'mean': 'category_mean', 'std' : 'category_std'})

,category,category_mean,category_std
0,abrasives supplier,4.304082,0.528346
1,accountant,4.561298,0.690987
2,accounting firm,4.607576,0.699079
3,acupuncture clinic,4.800866,0.281145
4,acupuncturist,4.805882,0.385867
...,...,...,...
1809,yoga instructor,4.610256,0.595065
1810,yoga studio,4.520263,0.609801
1811,youth clothing store,4.420270,0.242125
1812,youth organization,4.343169,0.579546


In [32]:

# Merge back into category data set
mcategory_df = category_df.merge(category_stats, on='category', how='inner') #what happens with a inner join.
mcategory_df = mcategory_df.rename(columns={
    'mean': 'category_mean',
    'std': 'category_std'
})
mcategory_df

,name,address,gmap_id,description,latitude,longitude,category,avg_rating,num_of_reviews,price,hours,MISC,state,relative_results,url,category_count,category_mean,category_std
0,Timewise Food Store,"Timewise Food Store, 1600 W Church St, Livings...",0x8638869e6b4e3529:0xe8d257447fe41672,None,30.713368,-94.954344,convenience store,4.8,4,None,"[[Thursday, Open 24 hours], [Friday, Open 24 h...","{'Service options': ['In-store shopping', 'Del...",Open 24 hours,"[0x863886bab3f9bb05:0x28a8062d0597dd34, 0x8638...",https://www.google.com/maps/place//data=!4m2!3...,9092.0,3.575594,0.771360
1,Goldstar Transit,"Goldstar Transit, 4415 W Dickson Ln, Little El...",0x864c3748dcc1c12d:0xbf904a61f262cf9b,None,33.159363,-96.975571,transportation service,4.5,4,None,"[[Thursday, 6AM–6PM], [Friday, 6AM–6PM], [Satu...",{'Accessibility': ['Wheelchair accessible entr...,Open ⋅ Closes 6PM,"[0x864c374855555555:0x3abb669a098bb235, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,1160.0,3.943362,0.792071
2,Walmart Pharmacy,"Walmart Pharmacy, 12220 FM 423, Frisco, TX 75033",0x864c3998b8d8dc83:0x57ffabe1e2322320,None,33.179867,-96.883691,pharmacy,3.3,24,$,"[[Thursday, 9AM–9PM], [Friday, 9AM–9PM], [Satu...","{'Service options': ['Curbside pickup', 'In-st...",Open ⋅ Closes 9PM,"[0x864c3999b29e291f:0x2d364c05e88eec13, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,1950.0,3.903282,0.822072
3,Walmart Pharmacy,"Walmart Pharmacy, 12220 FM 423, Frisco, TX 75033",0x864c3998b8d8dc83:0x57ffabe1e2322320,None,33.179867,-96.883691,drug store,3.3,24,$,"[[Thursday, 9AM–9PM], [Friday, 9AM–9PM], [Satu...","{'Service options': ['Curbside pickup', 'In-st...",Open ⋅ Closes 9PM,"[0x864c3999b29e291f:0x2d364c05e88eec13, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,2933.0,3.628981,0.741817
4,Walmart Pharmacy,"Walmart Pharmacy, 12220 FM 423, Frisco, TX 75033",0x864c3998b8d8dc83:0x57ffabe1e2322320,None,33.179867,-96.883691,medical supply store,3.3,24,$,"[[Thursday, 9AM–9PM], [Friday, 9AM–9PM], [Satu...","{'Service options': ['Curbside pickup', 'In-st...",Open ⋅ Closes 9PM,"[0x864c3999b29e291f:0x2d364c05e88eec13, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,456.0,3.657895,0.944378
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
952057,O'Reilly Auto Parts,"O'Reilly Auto Parts, 702 Tennessee Ave, Dalhar...",0x8705b6b8981c266d:0x5770fa9f1b6c0b1a,None,36.058740,-102.514025,car repair and maintenance,4.3,94,None,"[[Wednesday, 7AM–9PM], [Thursday, 7AM–9PM], [F...","{'Service options': ['Curbside pickup', 'In-st...",NaN,"[0x8705b6c7859a9041:0x9107ec50aa4d5609, 0x8705...",https://www.google.com/maps/place//data=!4m2!3...,3356.0,4.384058,0.444270
952058,O'Reilly Auto Parts,"O'Reilly Auto Parts, 702 Tennessee Ave, Dalhar...",0x8705b6b8981c266d:0x5770fa9f1b6c0b1a,None,36.058740,-102.514025,car stereo store,4.3,94,None,"[[Wednesday, 7AM–9PM], [Thursday, 7AM–9PM], [F...","{'Service options': ['Curbside pickup', 'In-st...",NaN,"[0x8705b6c7859a9041:0x9107ec50aa4d5609, 0x8705...",https://www.google.com/maps/place//data=!4m2!3...,1140.0,4.362368,0.304770
952059,O'Reilly Auto Parts,"O'Reilly Auto Parts, 702 Tennessee Ave, Dalhar...",0x8705b6b8981c266d:0x5770fa9f1b6c0b1a,None,36.058740,-102.514025,motorcycle parts store,4.3,94,None,"[[Wednesday, 7AM–9PM], [Thursday, 7AM–9PM], [F...","{'Service options': ['Curbside pickup', 'In-st...",NaN,"[0x8705b6c7859a9041:0x9107ec50aa4d5609, 0x8705...",https://www.google.com/maps/place//data=!4m2!3...,943.0,4.369671,0.241309
952060,O'Reilly Auto Parts,"O'Reilly Auto Parts, 702 Tennessee Ave, Dalhar...",0x8705b6b8981c266d:0x5770fa9f1b6c0b1a,None,36.058740,-102.514025,trailer supply store,4.3,94,None,"[[Wednesday, 7AM–9PM], [Thursday, 7AM–9PM], [F...","{'Service options': ['Curbside pickup', 'In-st...",NaN,"[0x8705b6c7859a9041:0x9107ec50aa4d5609, 0x8705...",https://www.google.com/maps/place//data=!4m2!3...,1465.0,4.227304,0.456606


In [33]:

# Normalized rating
mcategory_df['rating_zscore'] = (
    (mcategory_df['avg_rating'] - mcategory_df['category_mean']) /
    mcategory_df['category_std']
)
mcategory_df #now this is going to include repeats of companies when it is done

,name,address,gmap_id,description,latitude,longitude,category,avg_rating,num_of_reviews,price,hours,MISC,state,relative_results,url,category_count,category_mean,category_std,rating_zscore
0,Timewise Food Store,"Timewise Food Store, 1600 W Church St, Livings...",0x8638869e6b4e3529:0xe8d257447fe41672,None,30.713368,-94.954344,convenience store,4.8,4,None,"[[Thursday, Open 24 hours], [Friday, Open 24 h...","{'Service options': ['In-store shopping', 'Del...",Open 24 hours,"[0x863886bab3f9bb05:0x28a8062d0597dd34, 0x8638...",https://www.google.com/maps/place//data=!4m2!3...,9092.0,3.575594,0.771360,1.587334
1,Goldstar Transit,"Goldstar Transit, 4415 W Dickson Ln, Little El...",0x864c3748dcc1c12d:0xbf904a61f262cf9b,None,33.159363,-96.975571,transportation service,4.5,4,None,"[[Thursday, 6AM–6PM], [Friday, 6AM–6PM], [Satu...",{'Accessibility': ['Wheelchair accessible entr...,Open ⋅ Closes 6PM,"[0x864c374855555555:0x3abb669a098bb235, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,1160.0,3.943362,0.792071,0.702763
2,Walmart Pharmacy,"Walmart Pharmacy, 12220 FM 423, Frisco, TX 75033",0x864c3998b8d8dc83:0x57ffabe1e2322320,None,33.179867,-96.883691,pharmacy,3.3,24,$,"[[Thursday, 9AM–9PM], [Friday, 9AM–9PM], [Satu...","{'Service options': ['Curbside pickup', 'In-st...",Open ⋅ Closes 9PM,"[0x864c3999b29e291f:0x2d364c05e88eec13, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,1950.0,3.903282,0.822072,-0.733855
3,Walmart Pharmacy,"Walmart Pharmacy, 12220 FM 423, Frisco, TX 75033",0x864c3998b8d8dc83:0x57ffabe1e2322320,None,33.179867,-96.883691,drug store,3.3,24,$,"[[Thursday, 9AM–9PM], [Friday, 9AM–9PM], [Satu...","{'Service options': ['Curbside pickup', 'In-st...",Open ⋅ Closes 9PM,"[0x864c3999b29e291f:0x2d364c05e88eec13, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,2933.0,3.628981,0.741817,-0.443480
4,Walmart Pharmacy,"Walmart Pharmacy, 12220 FM 423, Frisco, TX 75033",0x864c3998b8d8dc83:0x57ffabe1e2322320,None,33.179867,-96.883691,medical supply store,3.3,24,$,"[[Thursday, 9AM–9PM], [Friday, 9AM–9PM], [Satu...","{'Service options': ['Curbside pickup', 'In-st...",Open ⋅ Closes 9PM,"[0x864c3999b29e291f:0x2d364c05e88eec13, 0x864c...",https://www.google.com/maps/place//data=!4m2!3...,456.0,3.657895,0.944378,-0.378974
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
952057,O'Reilly Auto Parts,"O'Reilly Auto Parts, 702 Tennessee Ave, Dalhar...",0x8705b6b8981c266d:0x5770fa9f1b6c0b1a,None,36.058740,-102.514025,car repair and maintenance,4.3,94,None,"[[Wednesday, 7AM–9PM], [Thursday, 7AM–9PM], [F...","{'Service options': ['Curbside pickup', 'In-st...",NaN,"[0x8705b6c7859a9041:0x9107ec50aa4d5609, 0x8705...",https://www.google.com/maps/place//data=!4m2!3...,3356.0,4.384058,0.444270,-0.189205
952058,O'Reilly Auto Parts,"O'Reilly Auto Parts, 702 Tennessee Ave, Dalhar...",0x8705b6b8981c266d:0x5770fa9f1b6c0b1a,None,36.058740,-102.514025,car stereo store,4.3,94,None,"[[Wednesday, 7AM–9PM], [Thursday, 7AM–9PM], [F...","{'Service options': ['Curbside pickup', 'In-st...",NaN,"[0x8705b6c7859a9041:0x9107ec50aa4d5609, 0x8705...",https://www.google.com/maps/place//data=!4m2!3...,1140.0,4.362368,0.304770,-0.204641
952059,O'Reilly Auto Parts,"O'Reilly Auto Parts, 702 Tennessee Ave, Dalhar...",0x8705b6b8981c266d:0x5770fa9f1b6c0b1a,None,36.058740,-102.514025,motorcycle parts store,4.3,94,None,"[[Wednesday, 7AM–9PM], [Thursday, 7AM–9PM], [F...","{'Service options': ['Curbside pickup', 'In-st...",NaN,"[0x8705b6c7859a9041:0x9107ec50aa4d5609, 0x8705...",https://www.google.com/maps/place//data=!4m2!3...,943.0,4.369671,0.241309,-0.288722
952060,O'Reilly Auto Parts,"O'Reilly Auto Parts, 702 Tennessee Ave, Dalhar...",0x8705b6b8981c266d:0x5770fa9f1b6c0b1a,None,36.058740,-102.514025,trailer supply store,4.3,94,None,"[[Wednesday, 7AM–9PM], [Thursday, 7AM–9PM], [F...","{'Service options': ['Curbside pickup', 'In-st...",NaN,"[0x8705b6c7859a9041:0x9107ec50aa4d5609, 0x8705...",https://www.google.com/maps/place/

*****#What we can do is group by gmap_id and then avg the z_scores per g_map into a df that only includes the avg z and .apply the confidence intervals to gmap_id*****

In [ ]:
#We will merge all of this back into texas metadata with an inner merge to add the new z-score and CI columns for the Tableu dataset.

mcategory_df['CI']= mcategory_df['gmap_id'].apply(individual_company_rating_confidence_interval)
mcategory_df #THIS TAKES TO LONG SPEED IT UP AND THEN MERGE BACK TO TEXAS METADATA

KeyboardInterrupt: 